# Data Preprocessing

### Imports

In [3]:
import pandas as pd
from pathlib import Path

### Defining the Target Country Set

Name key setting for initial screening of middle eastern coutnries

In [4]:
TARGET_CODES = [
    "BHR",  # Bahrain
    "KWT",  # Kuwait
    "OMN",  # Oman
    "QAT",  # Qatar
    "SAU",  # Saudi Arabia
    "ARE",  # United Arab Emirates
    "YEM",  # Yemen
    "IRQ",  # Iraq
    "JOR",  # Jordan
    "LBN",  # Lebanon
    "PSE",  # Palestine
    "SYR",  # Syria
    "EGY",  # Egypt
    "IRN",  # Iran
    "TUR",  # Turkey
    "CYP",  # Cyprus
]

# Clean display names for final output
COUNTRY_NAME_MAP = {
    "BHR": "Bahrain",
    "KWT": "Kuwait",
    "OMN": "Oman",
    "QAT": "Qatar",
    "SAU": "Saudi Arabia",
    "ARE": "United Arab Emirates",
    "YEM": "Yemen",
    "IRQ": "Iraq",
    "JOR": "Jordan",
    "LBN": "Lebanon",
    "PSE": "Palestine",
    "SYR": "Syria",
    "EGY": "Egypt",
    "IRN": "Iran",
    "TUR": "Turkey",
    "CYP": "Cyprus",
}

In [5]:
DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs")

OUTPUT_DIR.mkdir(exist_ok=True)

GDP_PATH = DATA_DIR / "gdp.csv"
CANCER_PATH = DATA_DIR / "cancer.csv"
OBESITY_PATH = DATA_DIR / "obesity.csv"
PHYSICAL_ACTIVITY_PATH = DATA_DIR / "activity.csv"

USE_SELECTED_COUNTRIES_ONLY = False
# True  = use only TARGET_CODES
# False = use all countries available in the cancer dataset

In [6]:
# GDP file has metadata rows at the top, so we skip the first 4 rows
gdp_df = pd.read_csv(GDP_PATH, skiprows=4)

cancer_df = pd.read_csv(CANCER_PATH)
obesity_df = pd.read_csv(OBESITY_PATH)
physical_df = pd.read_csv(PHYSICAL_ACTIVITY_PATH)

## Columns to be extracted.

Obesity

In [7]:
obesity_clean = obesity_df[
    [
        "SpatialDimValueCode",
        "Location",
        "FactValueNumeric",
        "FactValueNumericHigh",
        "FactValueNumericLow",
    ]
].copy()

obesity_clean = obesity_clean.rename(
    columns={
        "SpatialDimValueCode": "country_code",
        "Location": "obesity_country_name",
        "FactValueNumeric": "obesity_value",
        "FactValueNumericHigh": "obesity_high",
        "FactValueNumericLow": "obesity_low",
    }
)

# Keep only one row per country
obesity_clean = obesity_clean.drop_duplicates(subset=["country_code"])

Insuffecient Physcial activity

In [8]:
physical_clean = physical_df[
    [
        "SpatialDimValueCode",
        "Location",
        "FactValueNumeric",
        "FactValueNumericHigh",
        "FactValueNumericLow",
    ]
].copy()

physical_clean = physical_clean.rename(
    columns={
        "SpatialDimValueCode": "country_code",
        "Location": "inactivity_country_name",
        "FactValueNumeric": "inactivity_value",
        "FactValueNumericHigh": "inactivity_high",
        "FactValueNumericLow": "inactivity_low",
    }
)

# Keep only one row per country
physical_clean = physical_clean.drop_duplicates(subset=["country_code"])


GDP per Capita

In [9]:
gdp_clean = gdp_df[
    [
        "Country Code",
        "Country Name",
        "2022",
    ]
].copy()

gdp_clean = gdp_clean.rename(
    columns={
        "Country Code": "country_code",
        "Country Name": "gdp_country_name",
        "2022": "gdp_per_capita_2022",
    }
)

gdp_clean = gdp_clean.drop_duplicates(subset=["country_code"])

Cancer Data

In [10]:
cancer_clean = cancer_df[
    [
        "Alpha-3 code",
        "Label",
        "Number.1",
        "ASR (World)",
        "Crude rate",
    ]
].copy()

cancer_clean = cancer_clean.rename(
    columns={
        "Alpha-3 code": "country_code",
        "Label": "cancer_country_name",
        "Number.1": "new_cases",
        "ASR (World)": "asr_world",
        "Crude rate": "crude_rate",
    }
)

cancer_clean = cancer_clean.drop_duplicates(subset=["country_code"])

## Datat Merging

In [11]:
if USE_SELECTED_COUNTRIES_ONLY:
    # Use only the selected countries
    target_df = pd.DataFrame({"country_code": TARGET_CODES})

else:
    # Use all countries available in the cancer dataset
    # Cancer is the main focus of the project, so it is used as the base.
    target_df = cancer_clean[["country_code"]].drop_duplicates().copy()

In [12]:
merged_df = (
    target_df
    .merge(obesity_clean, on="country_code", how="left")
    .merge(physical_clean, on="country_code", how="left")
    .merge(gdp_clean, on="country_code", how="left")
    .merge(cancer_clean, on="country_code", how="left")
)

In [13]:
if USE_SELECTED_COUNTRIES_ONLY:
    merged_df.insert(
        0,
        "country_name",
        merged_df["country_code"].map(COUNTRY_NAME_MAP)
    )

else:
    merged_df.insert(
        0,
        "country_name",
        merged_df["cancer_country_name"]
    )

In [14]:
final_df = merged_df[
    [
        "country_name",
        "country_code",
        "obesity_value",
        "obesity_high",
        "obesity_low",
        "inactivity_value",
        "inactivity_high",
        "inactivity_low",
        "gdp_per_capita_2022",
        "asr_world",
        "new_cases",
        "crude_rate",
    ]
].copy()

In [15]:
# Check for missing values

print("Missing values per column:")
print(final_df.isna().sum())

print("\nCountries with any missing values:")
print(final_df[final_df.isna().any(axis=1)])


Missing values per column:
country_name            0
country_code            1
obesity_value           7
obesity_high            7
obesity_low             7
inactivity_value        8
inactivity_high         8
inactivity_low          8
gdp_per_capita_2022    10
asr_world               0
new_cases               0
crude_rate              0
dtype: int64

Countries with any missing values:
                               country_name country_code  obesity_value  \
41                                     Cuba          CUB          21.76   
51                                  Eritrea          ERI           4.83   
56                            French Guyana          GUF            NaN   
57                         French Polynesia          PYF          48.09   
66                       France, Guadeloupe          GLP            NaN   
67                                     Guam          GUM            NaN   
88   Korea, Democratic People's Republic of          PRK          10.80   
106         

In [16]:
# final_df = final_df.dropna().copy()

# print("\nFinal table after removing countries with missing values:")
# print(final_df)

In [17]:
# final_df.to_csv("final_merged_15_arab_countries.csv", index=False)

In [18]:
# Convert physical inactivity into physical activity


old_inactivity_value = final_df["inactivity_value"].copy()
old_inactivity_high = final_df["inactivity_high"].copy()
old_inactivity_low = final_df["inactivity_low"].copy()

# Convert main value
final_df["activity_value"] = 100 - old_inactivity_value

# Convert confidence interval correctly
# Important: high and low must swap after reversing
final_df["activity_high"] = 100 - old_inactivity_low
final_df["activity_low"] = 100 - old_inactivity_high

# Drop original inactivity columns
final_df = final_df.drop(
    columns=[
        "inactivity_value",
        "inactivity_high",
        "inactivity_low",
    ]
)

# Reorder final columns
final_df = final_df[
    [
        "country_name",
        "country_code",
        "obesity_value",
        "obesity_high",
        "obesity_low",
        "activity_value",
        "activity_high",
        "activity_low",
        "gdp_per_capita_2022",
        "asr_world",
        "new_cases",
        "crude_rate",
    ]
].copy()

print("\nFinal table after converting inactivity into activity:")
print(final_df)


Final table after converting inactivity into activity:
    country_name country_code  obesity_value  obesity_high  obesity_low  \
0    Afghanistan          AFG          19.22         22.20        16.33   
1        Albania          ALB          23.36         26.48        20.43   
2        Algeria          DZA          23.81         27.25        20.51   
3         Angola          AGO          11.47         16.03         7.63   
4     Azerbaijan          AZE          26.55         29.97        23.21   
..           ...          ...            ...           ...          ...   
181    Venezuela          VEN          22.72         25.69        19.78   
182        Samoa          WSM          62.43         67.69        56.94   
183        Yemen          YEM          13.65         17.49        10.45   
184       Zambia          ZMB          11.08         13.31         9.00   
185        Total          NaN            NaN           NaN          NaN   

     activity_value  activity_high  activit

In [ ]:
if USE_SELECTED_COUNTRIES_ONLY:
    output_path = OUTPUT_DIR / "preprocessed_middle_east_master.csv"
else:
    output_path = OUTPUT_DIR / "preprocessed_all_master.csv"

final_df.to_csv(output_path, index=False)

print("\nFinal cleaned dataset saved as:")
print(output_path)

print("\nFinal cleaned table:")
print(final_df)


Final cleaned dataset saved as:
outputs\preprocessed_all_master.csv

Final cleaned table:
    country_name country_code  obesity_value  obesity_high  obesity_low  \
0    Afghanistan          AFG          19.22         22.20        16.33   
1        Albania          ALB          23.36         26.48        20.43   
2        Algeria          DZA          23.81         27.25        20.51   
3         Angola          AGO          11.47         16.03         7.63   
4     Azerbaijan          AZE          26.55         29.97        23.21   
..           ...          ...            ...           ...          ...   
181    Venezuela          VEN          22.72         25.69        19.78   
182        Samoa          WSM          62.43         67.69        56.94   
183        Yemen          YEM          13.65         17.49        10.45   
184       Zambia          ZMB          11.08         13.31         9.00   
185        Total          NaN            NaN           NaN          NaN   

     act